# Fuzzy Search
Test fuzzy title matching using pg_trgm — both the raw function and the agent.

In [7]:
import sys
sys.path.insert(0, "..")

from movie_agent.search.fuzzy import fuzzy_search_title
from movie_agent.agent.fuzzy_search_agent import ask, ask_stream

## 1. Raw function: `fuzzy_search_title()`

In [9]:
# Exact title
results = fuzzy_search_title("spider man", top_k=3)

for movie in results:
    print(f"{movie['title']} (similarity: {movie['similarity']}) - Rating: {movie['vote_average']}")

Spider-Man (similarity: 1.0) - Rating: 6.8
Spider-Man 3 (similarity: 0.8462) - Rating: 5.9
Spider-Man 2 (similarity: 0.8462) - Rating: 6.7


In [10]:
# Misspelled title
results = fuzzy_search_title("Incpetion", top_k=3)

for movie in results:
    print(f"{movie['title']} (similarity: {movie['similarity']}) - Rating: {movie['vote_average']}")

In [11]:
# Partial title
results = fuzzy_search_title("Dark Knight", top_k=5)

for movie in results:
    print(f"{movie['title']} (similarity: {movie['similarity']})")
    print(f"   {movie['overview'][:100]}...")
    print()

The Dark Knight (similarity: 0.75)
   Batman raises the stakes in his war on crime. With the help of Lt. Jim Gordon and District Attorney ...



In [12]:
# Lowering threshold to get more results
results = fuzzy_search_title("Spiderman", top_k=5, threshold=0.2)

for movie in results:
    print(f"{movie['title']} (similarity: {movie['similarity']})")

Spider-Man (similarity: 0.6154)
Spider (similarity: 0.5455)
Spider-Man 2 (similarity: 0.5333)
Spider-Man 3 (similarity: 0.5333)
Superman (similarity: 0.3571)


## 2. Agent: `ask()`

In [13]:
result = ask("Tell me about the movie Interstellar")
print(result["answer"])

Here are the details for the movie **Interstellar**:

- **Rating**: 8.1
- **Release Date**: Not specified
- **Overview**: Interstellar chronicles the adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and embark on an interstellar journey to find a new home for humanity.

If you need more information or have specific questions about the movie, feel free to ask!


In [14]:
# Typo in title
result = ask("What is the movie Incpetion about?")
print(result["answer"])

The movie you're looking for is **Inception**. 

**Overview**: Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets, is offered a chance to regain his old life as part of a final job. Instead of stealing an idea, he must plant one, which is known as "inception."

**Rating**: 8.1

If you need more details or have any specific questions about the movie, feel free to ask!


## 3. Streaming: `ask_stream()`

In [15]:
print("=" * 60)
print("Query: Find the movie called the dark nite")
print("=" * 60)

for event in ask_stream("Find the movie called the dark nite"):
    if event["type"] == "tool_call":
        print(f"\n\U0001f6e0\ufe0f  TOOL CALL: {event['tool']}")
        print(f"   Input: {event['input']}")
    elif event["type"] == "tool_result":
        print(f"\n\U0001f4cb RESULT from {event['tool']}:")
        print(f"   {event['output'][:500]}")
    elif event["type"] == "token":
        print(event["content"], end="", flush=True)

print()

Query: Find the movie called the dark nite

🛠️  TOOL CALL: search_movie_by_title
   Input: {'title': 'the dark nite'}

📋 RESULT from search_movie_by_title:
   No movies found matching the title 'the dark nite'. Try a different spelling or a more complete title.

🛠️  TOOL CALL: search_movie_by_title
   Input: {'title': 'the dark knight'}

📋 RESULT from search_movie_by_title:
   1. **The Dark Knight** (match: 1.0)
   Rating: 8.2 | Released: None
   Batman raises the stakes in his war on crime. With the help of Lt. Jim Gordon and District Attorney Harvey Dent, Batman sets out to dismantle the rema...

2. **The Dark Knight Rises** (match: 0.7273)
   Rating: 7.6 | Released: None
   Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation and is...
I found a couple of movies that closely match your request:

1. **The Dark Knight** 
   - **Similarity Score:** 1.0
   - **Rating:** 8.2
   - **Overview:** Batm